# HotpotQA 多义词数据集 · Anchor-C/D/E/F 批量评测

加载 `hotpotqa_polysemy_dataset_build.ipynb` 产出的 gold 数据集，对每个词跑 **Anchor-C / Anchor-D / Anchor-E / Anchor-F**，并以数据集 gold 为真值汇总评测指标。

口径与 `hotpotqa_anchor_propagation_compare.ipynb` 一致：

- **方法消融阶梯（C→F）**，均以数据集存好的 LLM anchor（merge 标签）为锚点，在 span occurrence 的 embedding kNN 图上传播：
  - **C**：LLM anchor + plain kNN 多数票，无 CE margin 反对票、无 anchor center 检查（最激进，覆盖高、易扩散局部错误）。
  - **D**：同 C，但把 plain kNN 换成 mutual-kNN（图更稀疏、更保守，uncertain 更多）。
  - **E**：在 C/D 基础上加 cross-encoder margin 反对票（plain / mutual 各测一组）。
  - **F**：在 E 基础上再加 anchor center 检查与稀有类保守策略（plain / mutual 各测一组）。
  - 每个版本都跑 ce_fallback / re_llm 两种 uncertain 收尾，共 **12 个方法**（C×1 + D×1 + E×2 + F×2 个 knn 配置，再 ×2 收尾）。
- 每个词用数据集里存好的 candidate_bank + span embedding + anchor seed（merge 标签）复算，**无需重新编码 / 重新调 LLM**。
- C/D/E/F 都需要 Full cross-encoder（E/F 的 margin 反对票、各方法的 ce_fallback 收尾、以及 ce-only baseline），本 notebook 用存好的 `prompt_text` + 候选 `hypothesis` 现场跑 CE。
- 指标：**ce-only accuracy（纯 CE baseline，不经传播）**、overall / propagation-only accuracy、macro-F1、rare-class recall/precision、corrected CE error、newly introduced error、uncertain 比例、LLM/CE 调用计数。
- `ce_only_acc` 不依赖 anchor 传播，对同一词的 12 个方法相同，作为「只用 cross-encoder 分类能达到多少」的对照基线；传播相对它的增益/损失见 `corrected_ce_errors` / `newly_introduced_errors`。
- **propagation-only** 去掉 anchor 样本，才真实反映传播质量（anchor 标签复用 merge，必然偏高）。
- `re_llm` 收尾在本评测里等于命中 gold（上限估计），仅用于数 LLM 追加调用次数，看绝对精度请以 `ce_fallback` 为准。


In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Working directory: {REPO_ROOT}")

Working directory: /home/xiaoyue/LiteSemRAG


In [2]:
from collections import Counter
from pathlib import Path
import pickle
import re
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sentence_transformers import CrossEncoder
from sklearn.metrics import f1_score

from utils import extract_cross_encoder_scores

In [3]:
# =============================================================================
# 评测参数
# =============================================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 数据集（build notebook 的产物）---
# build notebook 现在输出带 LLM_RUN_TAG 前缀和时间戳的文件名，例如：
#   hotpotqa_polysemy_dataset_deepseek_v4_flash_top50_max100_20260605_142830.pkl
# 这里不再硬编码确切文件名，而是按 top{K}_max{M} 通配，自动选最新（时间戳最大）的一个。
TOP_K_WORDS = 50
MAX_OCCURRENCES_PER_WORD = 100
DATASET_DIR = REPO_ROOT / "data" / "polysemy_sem_eval"
DATASET_GLOB = f"hotpotqa_polysemy_dataset_*top{TOP_K_WORDS}_max{MAX_OCCURRENCES_PER_WORD}*.pkl"
# 复现某个具体数据集时，把文件名（或绝对路径）填到这里；None=按 DATASET_GLOB 自动选最新。
DATASET_PKL_OVERRIDE = None


def _dataset_version_sort_key(path):
    """优先按文件名尾部时间戳（YYYYMMDD_HHMMSS）排序，回退到 mtime。"""
    match = re.search(r"(\d{8}_\d{6})", path.stem)
    return (match.group(1) if match else "", path.stat().st_mtime)


def resolve_dataset_path():
    if DATASET_PKL_OVERRIDE:
        override = Path(DATASET_PKL_OVERRIDE)
        return override if override.is_absolute() else DATASET_DIR / override
    candidates = sorted(DATASET_DIR.glob(DATASET_GLOB), key=_dataset_version_sort_key)
    if not candidates:
        raise FileNotFoundError(
            f"在 {DATASET_DIR} 下找不到匹配 {DATASET_GLOB!r} 的数据集。"
            "请先运行 hotpotqa_polysemy_dataset_build.ipynb，或设置 DATASET_PKL_OVERRIDE。"
        )
    return candidates[-1]


DATASET_PKL_PATH = resolve_dataset_path()
# 输出文件跟随实际加载的数据集版本命名（去掉公共前缀），避免不同版本互相覆盖。
DATASET_TAG = DATASET_PKL_PATH.stem.replace("hotpotqa_polysemy_dataset_", "", 1)

# --- Full cross-encoder ---
CROSS_ENCODER_MODEL = "cross-encoder/nli-deberta-v3-large"
CROSS_ENCODER_BATCH_SIZE = 32

# --- 要评测的 anchor 版本（C/D/E/F）---
PROP_VERSIONS = ["C", "D", "E", "F"]
PROP_VERSION_FLAGS = {
    # C/D：最基础的 anchor 传播——不使用 CE margin 反对票，也不检查 anchor center，
    # 只靠 kNN 多数票。C 与 D 的唯一区别是图结构（见 PROP_VERSION_KNN_TYPES）。
    "C": dict(use_margin=False, use_center=False, rare_conservative=False),
    "D": dict(use_margin=False, use_center=False, rare_conservative=False),
    "E": dict(use_margin=True, use_center=False, rare_conservative=False),
    "F": dict(use_margin=True, use_center=True, rare_conservative=True),
}
# C 固定 plain kNN、D 固定 mutual kNN（二者只差图结构）；E/F 同时测 plain 与 mutual。
PROP_VERSION_KNN_TYPES = {
    "C": ["plain"],
    "D": ["mutual"],
    "E": ["plain", "mutual"],
    "F": ["plain", "mutual"],
}
PROP_UNCERTAIN_MODES = ["ce_fallback", "re_llm"]

# --- 传播超参（与 anchor_propagation_compare 一致）---
PROP_KNN_K = 8
PROP_VOTE_RATIO = 0.60
PROP_WEAK_VOTE_RATIO = 0.50
PROP_RARE_VOTE_RATIO = 0.80
PROP_HIGH_MARGIN = 1.5
PROP_CE_OPPOSE_GAP = 0.5
PROP_RARE_ANCHOR_THRESHOLD = 2
PROP_MAX_ROUNDS = 50

# --- 评测：gold 稀有类阈值 ---
RARE_GOLD_THRESHOLD = 5

# --- 输出 ---
RESULTS_DIR = REPO_ROOT / "data" / "polysemy_sem_eval"
PER_WORD_CSV_PATH = RESULTS_DIR / f"anchor_cdef_eval_per_word_{DATASET_TAG}.csv"
AGGREGATE_CSV_PATH = RESULTS_DIR / f"anchor_cdef_eval_aggregate_{DATASET_TAG}.csv"

pd.set_option("display.max_colwidth", None)
print(f"Device: {DEVICE}")
print(f"Dataset: {DATASET_PKL_PATH}")
print(f"Dataset tag: {DATASET_TAG}")
print(f"Versions: {PROP_VERSIONS} | knn: {PROP_VERSION_KNN_TYPES} | uncertain: {PROP_UNCERTAIN_MODES}")


def iter_anchor_prop_specs():
    for version in PROP_VERSIONS:
        for knn_type in PROP_VERSION_KNN_TYPES[version]:
            for uncertain_mode in PROP_UNCERTAIN_MODES:
                flags = dict(PROP_VERSION_FLAGS[version])
                flags["knn_type"] = knn_type
                yield version, knn_type, uncertain_mode, flags


def anchor_method_name(version, knn_type, uncertain_mode):
    return f"Anchor-{version}-{knn_type} [{uncertain_mode}]"

Device: cuda
Dataset: /home/xiaoyue/LiteSemRAG/data/polysemy_sem_eval/hotpotqa_polysemy_dataset_deepseek_v4_flash_top50_max100_20260605_142830.pkl
Dataset tag: deepseek_v4_flash_top50_max100_20260605_142830
Versions: ['C', 'D', 'E', 'F'] | knn: {'C': ['plain'], 'D': ['mutual'], 'E': ['plain', 'mutual'], 'F': ['plain', 'mutual']} | uncertain: ['ce_fallback', 're_llm']


In [4]:
if not DATASET_PKL_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATASET_PKL_PATH}. 请先运行 hotpotqa_polysemy_dataset_build.ipynb。"
    )

with DATASET_PKL_PATH.open("rb") as handle:
    dataset = pickle.load(handle)

word_entries = dataset["words"]
print(f"schema_version: {dataset['schema_version']} | words: {len(word_entries)}")
print(f"params: {dataset['params']}")
total_samples = sum(e["n_records"] for e in word_entries)
sense_dist = Counter(e["n_gold_senses"] for e in word_entries)
print(f"总样本: {total_samples} | gold 义项数分布: {dict(sorted(sense_dist.items()))}")

schema_version: 1 | words: 47
params: {'index_pkl_path': '/home/xiaoyue/LiteSemRAG/cache/hotpotqa_latest_framework_index/litesemrag_hotpotqa_500.pkl', 'scan_store_path': '/home/xiaoyue/LiteSemRAG/hotpot_QA_qwen_scan_cache/hotpot_phrase_token_scan_store_qwen_scan_v1_all.pkl', 'multi_sem_min_sem_count': 2, 'top_k_words': 50, 'max_occurrences_per_word': 100, 'min_occurrences_for_dataset': 20, 'query_kind': None, 'prompt_context_mode': 'sentence_neighbors', 'text_encoder': '/home/xiaoyue/ProtoGraphRAG/deberta-v3-large', 'embedding_method': 'LiteSemRAG span mean-pool from hidden_states[-2]', 'anchor_fraction': 0.15, 'anchor_fft_ratio': 0.7, 'anchor_min_count': 2, 'anchor_random_state': 42, 'wikidata_candidate_limit': 8, 'use_llm_wikidata': True, 'wikidata_llm_provider': 'deepseek', 'wikidata_llm_model': 'deepseek-v4-flash', 'gold_llm_provider': 'deepseek', 'gold_llm_model': 'deepseek-v4-flash', 'gold_llm_prompt_mode': 'batch', 'gold_llm_batch_size': 10}
总样本: 4645 | gold 义项数分布: {2: 17, 3: 21

In [5]:
cross_encoder_model = None


def ensure_cross_encoder_loaded(model_name: str):
    global cross_encoder_model
    if cross_encoder_model is None:
        cross_encoder_model = CrossEncoder(model_name)
        print(f"Loaded cross-encoder: {model_name}")
    return cross_encoder_model


def score_record_candidates(record, candidate_bank, model, batch_size=CROSS_ENCODER_BATCH_SIZE):
    pairs = [(record["prompt_text"], candidate["hypothesis"]) for candidate in candidate_bank]
    raw_scores = model.predict(pairs, batch_size=min(batch_size, len(pairs)), show_progress_bar=False)
    scores = extract_cross_encoder_scores(raw_scores, model)
    ranked_candidates = sorted(
        [{**candidate, "score": float(score)} for candidate, score in zip(candidate_bank, scores)],
        key=lambda item: item["score"], reverse=True,
    )
    return ranked_candidates


def classify_records_full_cross_encoder(records, candidate_bank, model, batch_size=CROSS_ENCODER_BATCH_SIZE):
    predictions = []
    for record in records:
        ranked = score_record_candidates(record, candidate_bank, model, batch_size=batch_size)
        top = ranked[0]
        predictions.append(
            {
                "record_index": record["record_index"],
                "assigned_description": top["description"],
                "predicted_entity_id": top["entity_id"],
                "predicted_label": top["label"],
                "ranked_candidates": ranked,
                "provenance": "full_cross_encoder",
            }
        )
    return predictions

In [6]:
# ====== anchor (mutual-)kNN 传播引擎（与 anchor_propagation_compare 一致）======

def _l2_normalize_rows(matrix):
    norms = np.clip(np.linalg.norm(matrix, axis=1, keepdims=True), 1e-12, None)
    return matrix / norms


def build_knn_graph(embeddings, knn_k, knn_type="mutual"):
    n = len(embeddings)
    normalized = _l2_normalize_rows(embeddings.astype(np.float32))
    sims = normalized @ normalized.T
    np.fill_diagonal(sims, -np.inf)
    k = min(int(knn_k), max(0, n - 1))
    knn_sets = [set() for _ in range(n)]
    if k > 0:
        for i in range(n):
            nbrs = np.argpartition(-sims[i], k - 1)[:k]
            knn_sets[i] = {int(j) for j in nbrs}

    adjacency = [set() for _ in range(n)]
    for i in range(n):
        for j in knn_sets[i]:
            if knn_type == "mutual":
                if i in knn_sets[j]:
                    adjacency[i].add(j)
                    adjacency[j].add(i)
            else:
                adjacency[i].add(j)
                adjacency[j].add(i)
    return adjacency


def _extract_ce_info(full_ce_predictions):
    ce_top1, ce_scores, ce_margin = {}, {}, {}
    for pred in full_ce_predictions:
        ranked = pred["ranked_candidates"]
        record_index = pred["record_index"]
        ce_top1[record_index] = ranked[0]["description"]
        ce_scores[record_index] = {c["description"]: float(c["score"]) for c in ranked}
        if len(ranked) >= 2:
            ce_margin[record_index] = float(ranked[0]["score"]) - float(ranked[1]["score"])
        else:
            ce_margin[record_index] = float("inf")
    return ce_top1, ce_scores, ce_margin


def run_anchor_propagation(
    records, full_ce_predictions, gold_label_by_index, anchor_positions, *,
    anchor_label_by_index=None, relabel_by_index=None, version, knn_type,
    use_margin, use_center, rare_conservative, uncertain_mode,
    knn_k=PROP_KNN_K, vote_ratio=PROP_VOTE_RATIO, weak_vote_ratio=PROP_WEAK_VOTE_RATIO,
    rare_vote_ratio=PROP_RARE_VOTE_RATIO, high_margin=PROP_HIGH_MARGIN,
    ce_oppose_gap=PROP_CE_OPPOSE_GAP, rare_anchor_threshold=PROP_RARE_ANCHOR_THRESHOLD,
    max_rounds=PROP_MAX_ROUNDS,
):
    n = len(records)
    embeddings = np.stack([r["embedding"] for r in records]).astype(np.float32)
    normalized = _l2_normalize_rows(embeddings)
    record_index_of = {pos: records[pos]["record_index"] for pos in range(n)}

    ce_top1, ce_scores, ce_margin = _extract_ce_info(full_ce_predictions)
    adjacency = build_knn_graph(embeddings, knn_k, knn_type=knn_type)

    anchor_label_source = anchor_label_by_index or gold_label_by_index
    relabel_source = relabel_by_index or gold_label_by_index
    requested_anchor_set = set(int(p) for p in anchor_positions)
    anchor_set = {pos for pos in requested_anchor_set if record_index_of[pos] in anchor_label_source}
    missing_anchor_record_indices = sorted(
        record_index_of[pos] for pos in requested_anchor_set if record_index_of[pos] not in anchor_label_source
    )
    labels = {}
    provenance = {}
    for pos in anchor_set:
        labels[pos] = anchor_label_source[record_index_of[pos]]
        provenance[pos] = "llm_anchor"

    anchor_class_counts = Counter(labels[pos] for pos in anchor_set)
    rare_classes = {desc for desc, cnt in anchor_class_counts.items() if cnt <= int(rare_anchor_threshold)}
    class_centers = {}
    for desc in anchor_class_counts:
        members = [pos for pos in anchor_set if labels[pos] == desc]
        center = normalized[members].mean(axis=0)
        center = center / max(np.linalg.norm(center), 1e-12)
        class_centers[desc] = center
    center_descs = list(class_centers.keys())
    center_matrix = (
        np.stack([class_centers[d] for d in center_descs]).astype(np.float32) if center_descs else None
    )

    def nearest_center_class(pos):
        if center_matrix is None:
            return None
        sims = center_matrix @ normalized[pos]
        return center_descs[int(np.argmax(sims))]

    def ce_opposes(pos, candidate_desc):
        if not use_margin:
            return False
        record_index = record_index_of[pos]
        top1 = ce_top1[record_index]
        if top1 == candidate_desc:
            return False
        if ce_margin[record_index] < high_margin:
            return False
        scores = ce_scores[record_index]
        return scores.get(candidate_desc, -float("inf")) < scores[top1] - ce_oppose_gap

    for _ in range(int(max_rounds)):
        snapshot = dict(labels)
        new_labels = {}
        for pos in range(n):
            if pos in labels:
                continue
            labeled_neighbors = [j for j in adjacency[pos] if j in snapshot]
            if not labeled_neighbors:
                continue
            votes = Counter(snapshot[j] for j in labeled_neighbors)
            maj, maj_count = votes.most_common(1)[0]
            ratio = maj_count / len(labeled_neighbors)

            is_rare = maj in rare_classes
            effective_vote_ratio = rare_vote_ratio if (rare_conservative and is_rare) else vote_ratio
            record_index = record_index_of[pos]
            opposes = ce_opposes(pos, maj)
            center_ok = use_center and (nearest_center_class(pos) == maj)

            chosen, prov = None, None
            if ratio >= effective_vote_ratio:
                if opposes:
                    if center_ok:
                        chosen, prov = maj, "center_override"
                elif rare_conservative and is_rare and use_center and not center_ok and maj != ce_top1[record_index]:
                    pass
                else:
                    prov = "neighbor_ce_agree" if maj == ce_top1[record_index] else "neighbor_override"
                    chosen = maj
            else:
                if use_center and center_ok and ratio >= weak_vote_ratio and not opposes:
                    chosen, prov = maj, "center_support"

            if chosen is not None:
                new_labels[pos] = (chosen, prov)

        if not new_labels:
            break
        for pos, (lab, prov) in new_labels.items():
            labels[pos] = lab
            provenance[pos] = prov

    uncertain_positions = set()
    for pos in range(n):
        if pos in labels:
            continue
        record_index = record_index_of[pos]
        if use_margin and ce_margin[record_index] >= high_margin:
            labels[pos] = ce_top1[record_index]
            provenance[pos] = "ce_high_margin"
        else:
            uncertain_positions.add(pos)
            provenance[pos] = "uncertain"

    llm_calls = len(anchor_set)
    for pos in uncertain_positions:
        record_index = record_index_of[pos]
        if uncertain_mode == "ce_fallback":
            labels[pos] = ce_top1[record_index]
            provenance[pos] = "uncertain_ce_fallback"
        elif uncertain_mode == "re_llm":
            if record_index not in relabel_source:
                raise ValueError(f"Missing re-LLM label for record_index={record_index}")
            labels[pos] = relabel_source[record_index]
            provenance[pos] = "uncertain_re_llm"
            llm_calls += 1
        else:
            raise ValueError("uncertain_mode must be 'ce_fallback' or 're_llm'")

    ce_calls = n if (use_margin or uncertain_mode == "ce_fallback") else 0

    pred_by_index = {record_index_of[pos]: labels[pos] for pos in range(n)}
    uncertain_index_set = {record_index_of[pos] for pos in uncertain_positions}
    anchor_index_set = {record_index_of[pos] for pos in anchor_set}

    return {
        "version": version,
        "knn_type": knn_type,
        "uncertain_mode": uncertain_mode,
        "pred_by_index": pred_by_index,
        "anchor_index_set": anchor_index_set,
        "uncertain_index_set": uncertain_index_set,
        "rare_classes": sorted(rare_classes),
        "missing_anchor_record_indices": missing_anchor_record_indices,
        "provenance_counts": dict(Counter(provenance.values())),
        "llm_anchor_labels": len(anchor_set),
        "llm_extra_labels": llm_calls - len(anchor_set),
        "llm_calls": llm_calls,
        "ce_calls": ce_calls,
        "n_records": n,
    }

In [7]:
# ====== 评测指标（与 anchor_propagation_compare 一致）======

def to_label_dict(assignments, key="assigned_description"):
    return {a["record_index"]: a[key] for a in assignments}


def rare_gold_classes_from_gold(gold_by_index, threshold=RARE_GOLD_THRESHOLD):
    counts = Counter(gold_by_index.values())
    return {desc for desc, cnt in counts.items() if cnt <= int(threshold)}


def evaluate_method(
    method_name, pred_by_index, gold_by_index, *, ce_top1_by_index,
    anchor_index_set=frozenset(), uncertain_index_set=frozenset(),
    rare_gold_classes=frozenset(), llm_anchor_labels=None, llm_extra_labels=None, ce_calls=None,
):
    idxs = [i for i in gold_by_index if i in pred_by_index]
    y_true = [gold_by_index[i] for i in idxs]
    y_pred = [pred_by_index[i] for i in idxs]
    total = len(idxs)
    overall_correct = sum(1 for i in idxs if pred_by_index[i] == gold_by_index[i])
    overall_acc = overall_correct / total if total else 0.0

    # 纯 cross-encoder baseline：所有样本直接取 CE top1（不经过 anchor 传播）的准确率。
    # 对同一个词的 12 个方法该值相同；它衡量“只用 CE 分类”能达到多少，作为传播的对照基线。
    ce_only_correct = sum(1 for i in idxs if ce_top1_by_index[i] == gold_by_index[i])
    ce_only_acc = ce_only_correct / total if total else 0.0

    prop_idxs = [i for i in idxs if i not in anchor_index_set]
    prop_correct = sum(1 for i in prop_idxs if pred_by_index[i] == gold_by_index[i])
    prop_acc = prop_correct / len(prop_idxs) if prop_idxs else float("nan")

    # 排除 anchor 且排除 uncertain：只统计传播阶段真正确定下来的样本。
    # 这部分标签不受 fallback 方式影响，最能反映传播本身的质量（ce_fallback 与 re_llm 该列相同）。
    prop_pure_idxs = [i for i in prop_idxs if i not in uncertain_index_set]
    prop_pure_correct = sum(1 for i in prop_pure_idxs if pred_by_index[i] == gold_by_index[i])
    prop_acc_excl_uncertain = prop_pure_correct / len(prop_pure_idxs) if prop_pure_idxs else float("nan")

    gold_classes = sorted(set(y_true))
    macro_f1 = f1_score(y_true, y_pred, labels=gold_classes, average="macro", zero_division=0)

    rare = set(rare_gold_classes)
    rare_gold_total = sum(1 for i in idxs if gold_by_index[i] in rare)
    rare_gold_hit = sum(1 for i in idxs if gold_by_index[i] in rare and pred_by_index[i] == gold_by_index[i])
    rare_pred_total = sum(1 for i in idxs if pred_by_index[i] in rare)
    rare_pred_hit = sum(1 for i in idxs if pred_by_index[i] in rare and pred_by_index[i] == gold_by_index[i])
    rare_recall = rare_gold_hit / rare_gold_total if rare_gold_total else float("nan")
    rare_precision = rare_pred_hit / rare_pred_total if rare_pred_total else float("nan")

    corrected_ce_errors = sum(
        1 for i in idxs if ce_top1_by_index[i] != gold_by_index[i] and pred_by_index[i] == gold_by_index[i]
    )
    newly_introduced_errors = sum(
        1 for i in idxs if ce_top1_by_index[i] == gold_by_index[i] and pred_by_index[i] != gold_by_index[i]
    )
    uncertain_ratio = len(uncertain_index_set) / total if total else 0.0

    return {
        "method": method_name,
        "total": total,
        "ce_only_acc": ce_only_acc,
        "overall_acc": overall_acc,
        "propagation_acc": prop_acc,
        "propagation_acc_excl_uncertain": prop_acc_excl_uncertain,
        "macro_f1": macro_f1,
        "rare_recall": rare_recall,
        "rare_precision": rare_precision,
        "corrected_ce_errors": corrected_ce_errors,
        "newly_introduced_errors": newly_introduced_errors,
        "uncertain_ratio": uncertain_ratio,
        "llm_anchor_labels": llm_anchor_labels,
        "llm_extra_labels": llm_extra_labels,
        "ce_calls": ce_calls,
    }

In [8]:
def reconstruct_records(entry):
    """从数据集样本重建传播所需的 records（位置下标==record_index）。"""
    samples = sorted(entry["samples"], key=lambda s: s["record_index"])
    records = []
    for pos, s in enumerate(samples):
        if s["record_index"] != pos:
            raise ValueError(
                f"word={entry['word']!r}: record_index {s['record_index']} != position {pos}; "
                "数据集样本顺序与下标不一致。"
            )
        records.append(
            {
                "record_index": s["record_index"],
                "embedding": np.asarray(s["embedding"], dtype=np.float32),
                "prompt_text": s["prompt_text"],
                "matched_text": s.get("matched_text"),
                "context_text": s.get("context_text"),
            }
        )
    gold_by_index = {s["record_index"]: s["gold_description"] for s in samples}
    return records, gold_by_index


def evaluate_word(entry, cross_encoder):
    records, gold_by_index = reconstruct_records(entry)
    candidate_bank = entry["candidate_bank"]
    anchor_positions = entry["anchor_positions"]
    anchor_label_by_index = entry.get("anchor_label_by_index") or {}

    full_ce_predictions = classify_records_full_cross_encoder(records, candidate_bank, cross_encoder)
    ce_top1_by_index = to_label_dict(full_ce_predictions)
    rare_gold = rare_gold_classes_from_gold(gold_by_index)

    rows = []
    runs = {}
    for version, knn_type, umode, flags in iter_anchor_prop_specs():
        result = run_anchor_propagation(
            records, full_ce_predictions, gold_by_index, anchor_positions,
            anchor_label_by_index=anchor_label_by_index, relabel_by_index=gold_by_index,
            version=version, uncertain_mode=umode, **flags,
        )
        method = anchor_method_name(version, knn_type, umode)
        metrics = evaluate_method(
            method, result["pred_by_index"], gold_by_index, ce_top1_by_index=ce_top1_by_index,
            anchor_index_set=result["anchor_index_set"], uncertain_index_set=result["uncertain_index_set"],
            rare_gold_classes=rare_gold, llm_anchor_labels=result["llm_anchor_labels"],
            llm_extra_labels=result["llm_extra_labels"], ce_calls=result["ce_calls"],
        )
        metrics = {"word": entry["word"], "n_gold_senses": entry["n_gold_senses"], **metrics}
        rows.append(metrics)
        runs[method] = {
            "result": result,
            "gold_by_index": gold_by_index,
            "ce_top1_by_index": ce_top1_by_index,
            "rare_gold": rare_gold,
        }
    return rows, runs


print("evaluate_word 已就绪。")

evaluate_word 已就绪。


In [9]:
cross_encoder = ensure_cross_encoder_loaded(CROSS_ENCODER_MODEL)

per_word_rows = []
# 全局池化（key/label 都按 word 命名空间化），用于 micro 指标的一次性 evaluate_method。
pooled = {
    method: {"pred": {}, "gold": {}, "ce_top1": {}, "anchor": set(), "uncertain": set(),
             "rare": set(), "llm_anchor": 0, "llm_extra": 0, "ce_calls": 0}
    for (v, k, u, _) in iter_anchor_prop_specs()
    for method in [anchor_method_name(v, k, u)]
}

progress = display("Evaluating 0/0 words", display_id=True)
start = time.time()
for wi, entry in enumerate(word_entries, start=1):
    rows, runs = evaluate_word(entry, cross_encoder)
    per_word_rows.extend(rows)

    w = entry["word"]
    for method, run in runs.items():
        result = run["result"]
        gold = run["gold_by_index"]
        ce_top1 = run["ce_top1_by_index"]
        rare = run["rare_gold"]
        agg = pooled[method]
        for ri, gdesc in gold.items():
            key = f"{w}#{ri}"
            agg["pred"][key] = f"{w}\t{result['pred_by_index'][ri]}"
            agg["gold"][key] = f"{w}\t{gdesc}"
            agg["ce_top1"][key] = f"{w}\t{ce_top1[ri]}"
            if ri in result["anchor_index_set"]:
                agg["anchor"].add(key)
            if ri in result["uncertain_index_set"]:
                agg["uncertain"].add(key)
        for desc in rare:
            agg["rare"].add(f"{w}\t{desc}")
        agg["llm_anchor"] += result["llm_anchor_labels"]
        agg["llm_extra"] += result["llm_extra_labels"]
        agg["ce_calls"] += result["ce_calls"]

    progress.update(
        f"Evaluating {wi}/{len(word_entries)} words | last={w!r} "
        f"(senses={entry['n_gold_senses']}) | {time.time()-start:.0f}s"
    )

per_word_df = pd.DataFrame(per_word_rows)
print(f"完成：{len(word_entries)} 词 × {len(pooled)} 方法 = {len(per_word_df)} 行。")

Loaded cross-encoder: cross-encoder/nli-deberta-v3-large


"Evaluating 47/47 words | last='performance' (senses=2) | 167s"

完成：47 词 × 12 方法 = 564 行。


In [10]:
metric_cols = [
    "ce_only_acc", "overall_acc", "propagation_acc", "propagation_acc_excl_uncertain",
    "macro_f1", "rare_recall", "rare_precision",
    "corrected_ce_errors", "newly_introduced_errors", "uncertain_ratio",
]

# --- macro：各词指标的简单平均（nanmean，处理无 propagation/rare 的词）---
macro_rows = []
for method, grp in per_word_df.groupby("method"):
    row = {"method": method, "aggregate": "macro(mean over words)", "n_words": len(grp)}
    for col in metric_cols:
        row[col] = float(np.nanmean(grp[col].to_numpy(dtype=float)))
    row["llm_anchor_labels"] = int(grp["llm_anchor_labels"].sum())
    row["llm_extra_labels"] = int(grp["llm_extra_labels"].sum())
    row["ce_calls"] = int(grp["ce_calls"].sum())
    macro_rows.append(row)
macro_df = pd.DataFrame(macro_rows)

# --- micro：所有样本池化后一次性计算（仍保留，用于写入汇总 CSV）---
micro_rows = []
for method, agg in pooled.items():
    m = evaluate_method(
        method, agg["pred"], agg["gold"], ce_top1_by_index=agg["ce_top1"],
        anchor_index_set=agg["anchor"], uncertain_index_set=agg["uncertain"],
        rare_gold_classes=agg["rare"], llm_anchor_labels=agg["llm_anchor"],
        llm_extra_labels=agg["llm_extra"], ce_calls=agg["ce_calls"],
    )
    m["aggregate"] = "micro(pooled)"
    micro_rows.append(m)
micro_df = pd.DataFrame(micro_rows)

display_cols = ["method", "aggregate"] + metric_cols + ["llm_anchor_labels", "llm_extra_labels", "ce_calls"]
# 整个数据集的总结只展示 macro（按词平均）；micro 仍计算并写入 CSV。
print("===== macro（按词平均）=====")
display(macro_df[[c for c in display_cols if c in macro_df.columns]].sort_values("method").reset_index(drop=True))
print(
    "提示：ce_only_acc 是纯 cross-encoder baseline（不经传播，12 个方法相同）；"
    "propagation_acc 含 uncertain 收尾（受 fallback 影响）；"
    "propagation_acc_excl_uncertain 排除 anchor+uncertain，只看传播本身、不受 fallback 影响"
    "（ce_fallback 与 re_llm 该列相同）；"
    "re_llm 的 overall/propagation 为上限估计（uncertain 直接命中 gold），比较真实精度请看 ce_fallback。"
)

===== macro（按词平均）=====


,method,aggregate,ce_only_acc,overall_acc,propagation_acc,propagation_acc_excl_uncertain,macro_f1,rare_recall,rare_precision,corrected_ce_errors,newly_introduced_errors,uncertain_ratio,llm_anchor_labels,llm_extra_labels,ce_calls
0,Anchor-C-plain [ce_fallback],macro(mean over words),0.637395,0.799338,0.780697,0.781036,0.640712,0.559167,0.390500,25.659574,9.595745,0.030747,677,0,4645
1,Anchor-C-plain [re_llm],macro(mean over words),0.637395,0.810224,0.793437,0.781036,0.653603,0.559167,0.432516,26.744681,9.595745,0.030747,677,143,0
2,Anchor-D-mutual [ce_fallback],macro(mean over words),0.637395,0.785734,0.763701,0.784597,0.653400,0.591667,0.563580,23.191489,8.446809,0.118375,677,0,4645
3,Anchor-D-mutual [re_llm],macro(mean over words),0.637395,0.839438,0.825256,0.784597,0.728395,0.670000,0.789496,28.531915,8.446809,0.118375,677,552,0
4,Anchor-E-mutual [ce_fallback],macro(mean over words),0.637395,0.792347,0.771423,0.794381,0.661388,0.641667,0.567284,22.914894,7.531915,0.113864,677,0,4645
5,Anchor-E-mutual [re_llm],macro(mean over words),0.637395,0.846703,0.833742,0.794381,0.736517,0.720000,0.779692,28.319149,7.531915,0.113864,677,531,4645
6,Anchor-E-plain [ce_fallback],macro(mean over words),0.637395,0.805909,0.788396,0.790217,0.648603,0.584167,0.381589,25.361702,8.659574,0.028705,677,0,4645
7,Anchor-E-plain [re_llm],macro(mean over words),0.637395,0.817008,0.801387,0.790217,0.661181,0.584167,0.423606,26.468085,8.659574,0.028705,677,134,4645
8,Anchor-F-mutual [ce_fallback],macro(mean over words),0.637395,0.791330,0.770255,0.792324,0.661371,0.616667,0.568210,23.085106,7.808511,0.114676,677,0,4645
9,Anchor-F-mutual [re_llm],macro(mean over words),0.637395,0.845891,0.832817,0.792324,0.736342,0.670000,0.782353,28.510638,7.808511,0.114676,677,535,4645


提示：ce_only_acc 是纯 cross-encoder baseline（不经传播，12 个方法相同）；propagation_acc 含 uncertain 收尾（受 fallback 影响）；propagation_acc_excl_uncertain 排除 anchor+uncertain，只看传播本身、不受 fallback 影响（ce_fallback 与 re_llm 该列相同）；re_llm 的 overall/propagation 为上限估计（uncertain 直接命中 gold），比较真实精度请看 ce_fallback。


In [11]:
# ====== 按单词排名：每个词取 overall_acc 最高的方法作为该词最终准确率 ======
# ce_only_acc 对同一词的 12 个方法相同，直接取最佳行上的值即可。
# overall_acc 并列时 idxmax 取首个出现的方法。
best_rows = per_word_df.loc[per_word_df.groupby("word")["overall_acc"].idxmax()]
best_per_word = (
    best_rows[["word", "overall_acc", "ce_only_acc", "method"]]
    .sort_values("overall_acc", ascending=False)
    .reset_index(drop=True)
    .rename(
        columns={
            "word": "单词",
            "overall_acc": "最高准确率",
            "ce_only_acc": "CrossEncoder准确率",
            "method": "最佳方法",
        }
    )
)
best_per_word.insert(0, "排名", range(1, len(best_per_word) + 1))

print(f"共 {len(best_per_word)} 个单词，按各词最高 overall_acc 从高到低排名：")
display(best_per_word)

共 47 个单词，按各词最高 overall_acc 从高到低排名：


,排名,单词,最高准确率,CrossEncoder准确率,最佳方法
0,1,season,0.990000,0.990000,Anchor-D-mutual [re_llm]
1,2,organization,0.989362,0.414894,Anchor-C-plain [ce_fallback]
2,3,competition,0.980000,0.970000,Anchor-C-plain [ce_fallback]
3,4,point,0.980000,0.480000,Anchor-D-mutual [re_llm]
4,5,release,0.980000,0.270000,Anchor-D-mutual [re_llm]
5,6,family,0.980000,0.700000,Anchor-F-mutual [re_llm]
6,7,color,0.978723,0.914894,Anchor-F-mutual [ce_fallback]
7,8,club,0.970000,0.700000,Anchor-F-plain [ce_fallback]
8,9,head,0.970000,0.160000,Anchor-D-mutual [re_llm]
9,10,race,0.970000,0.930000,Anchor-C-plain [ce_fallback]


In [12]:
# ====== Top-30 词的 macro 统计（mean ± std）======
# 排名口径与 best_per_word 完全一致：每词取 overall_acc 最高的方法行，再按 overall_acc 降序。
# 这里从 per_word_df 重取“带全列”的最佳行，以便拿到 propagation_acc_excl_uncertain。
TOP_N_WORDS_FOR_STATS = 30

best_rows_full = (
    per_word_df.loc[per_word_df.groupby("word")["overall_acc"].idxmax()]
    .sort_values("overall_acc", ascending=False)
    .reset_index(drop=True)
)
n_top = min(TOP_N_WORDS_FOR_STATS, len(best_rows_full))
top_rows = best_rows_full.head(n_top)

# (展示名, per_word_df 中的列)；每个词贡献一个数（即该词最佳方法行上的值），再 macro 平均。
stat_specs = [
    ("最高准确率(overall_acc)", "overall_acc"),
    ("propagation_acc_excl_uncertain", "propagation_acc_excl_uncertain"),
    ("CrossEncoder准确率(ce_only_acc)", "ce_only_acc"),
]

stat_rows = []
for label, col in stat_specs:
    vals = top_rows[col].to_numpy(dtype=float)
    n_valid = int(np.sum(~np.isnan(vals)))
    stat_rows.append(
        {
            "指标": label,
            "词数": n_valid,
            "平均值(macro)": float(np.nanmean(vals)),
            "标准差(ddof=1)": float(np.nanstd(vals, ddof=1)),
        }
    )

top_word_stats = pd.DataFrame(stat_rows)
print(f"对 overall_acc 排名前 {n_top} 个词做 macro 统计（每词取其最佳方法行的值）：")
display(top_word_stats)

对 overall_acc 排名前 30 个词做 macro 统计（每词取其最佳方法行的值）：


,指标,词数,平均值(macro),标准差(ddof=1)
0,最高准确率(overall_acc),30,0.936677,0.041569
1,propagation_acc_excl_uncertain,30,0.892257,0.175820
2,CrossEncoder准确率(ce_only_acc),30,0.664894,0.256774


In [13]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
per_word_df.to_csv(PER_WORD_CSV_PATH, index=False)

aggregate_df = pd.concat([micro_df, macro_df], ignore_index=True)
aggregate_df = aggregate_df[[c for c in display_cols if c in aggregate_df.columns]]
aggregate_df.to_csv(AGGREGATE_CSV_PATH, index=False)

print(f"已保存 per-word 指标: {PER_WORD_CSV_PATH}")
print(f"已保存汇总指标: {AGGREGATE_CSV_PATH}")
display(aggregate_df.sort_values(["aggregate", "method"]).reset_index(drop=True))

已保存 per-word 指标: /home/xiaoyue/LiteSemRAG/data/polysemy_sem_eval/anchor_cdef_eval_per_word_deepseek_v4_flash_top50_max100_20260605_142830.csv
已保存汇总指标: /home/xiaoyue/LiteSemRAG/data/polysemy_sem_eval/anchor_cdef_eval_aggregate_deepseek_v4_flash_top50_max100_20260605_142830.csv


,method,aggregate,ce_only_acc,overall_acc,propagation_acc,propagation_acc_excl_uncertain,macro_f1,rare_recall,rare_precision,corrected_ce_errors,newly_introduced_errors,uncertain_ratio,llm_anchor_labels,llm_extra_labels,ce_calls
0,Anchor-C-plain [ce_fallback],macro(mean over words),0.637395,0.799338,0.780697,0.781036,0.640712,0.559167,0.390500,25.659574,9.595745,0.030747,677,0,4645
1,Anchor-C-plain [re_llm],macro(mean over words),0.637395,0.810224,0.793437,0.781036,0.653603,0.559167,0.432516,26.744681,9.595745,0.030747,677,143,0
2,Anchor-D-mutual [ce_fallback],macro(mean over words),0.637395,0.785734,0.763701,0.784597,0.653400,0.591667,0.563580,23.191489,8.446809,0.118375,677,0,4645
3,Anchor-D-mutual [re_llm],macro(mean over words),0.637395,0.839438,0.825256,0.784597,0.728395,0.670000,0.789496,28.531915,8.446809,0.118375,677,552,0
4,Anchor-E-mutual [ce_fallback],macro(mean over words),0.637395,0.792347,0.771423,0.794381,0.661388,0.641667,0.567284,22.914894,7.531915,0.113864,677,0,4645
5,Anchor-E-mutual [re_llm],macro(mean over words),0.637395,0.846703,0.833742,0.794381,0.736517,0.720000,0.779692,28.319149,7.531915,0.113864,677,531,4645
6,Anchor-E-plain [ce_fallback],macro(mean over words),0.637395,0.805909,0.788396,0.790217,0.648603,0.584167,0.381589,25.361702,8.659574,0.028705,677,0,4645
7,Anchor-E-plain [re_llm],macro(mean over words),0.637395,0.817008,0.801387,0.790217,0.661181,0.584167,0.423606,26.468085,8.659574,0.028705,677,134,4645
8,Anchor-F-mutual [ce_fallback],macro(mean over words),0.637395,0.791330,0.770255,0.792324,0.661371,0.616667,0.568210,23.085106,7.808511,0.114676,677,0,4645
9,Anchor-F-mutual [re_llm],macro(mean over words),0.637395,0.845891,0.832817,0.792324,0.736342,0.670000,0.782353,28.510638,7.808511,0.114676,677,535,4645
